# Model A: Fire Detection Training

**Architecture:** YOLOX-M  
**Classes:** `fire`, `smoke` (2 classes)  
**Dataset:** ~122,525 images (merged from multiple sources)  
**Input Size:** 640x640  

## Target Metrics

| Metric | Target | Min Acceptable |
|--------|--------|-----------|
| Precision | >= 0.90 | >= 0.85 |
| Recall | >= 0.88 | >= 0.82 |
| mAP@0.5 | >= 0.85 | >= 0.75 |
| FP Rate | < 3% | < 5% |
| FN Rate | < 5% | < 8% |

## Notes

- Large dataset with good class coverage
- Standard augmentation pipeline (mosaic, mixup, HSV shifts)
- Fire and smoke can co-occur in the same frame — multi-label detection
- Training config: `features/safety-fire_detection/configs/02_training.yaml`
- Data config: `features/safety-fire_detection/configs/01_data.yaml`

In [ ]:
import sys
from pathlib import Path

# Add pipeline root to path so we can import our modules
PIPELINE_ROOT = Path("..").resolve()
sys.path.insert(0, str(PIPELINE_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from utils.config import load_config
from utils.device import get_device, set_seed

# Load configs
train_config = load_config(PIPELINE_ROOT / "configs" / "training" / "fire.yaml")
data_config = load_config(PIPELINE_ROOT / "configs" / "data" / "fire.yaml")

# Reproducibility
set_seed(train_config.get("seed", 42))
device = get_device()
print(f"Device: {device}")
print(f"Pipeline root: {PIPELINE_ROOT}")

## Data Preview

Load the fire dataset and inspect class distribution and sample images.

In [ ]:
from data_transforms.dataset import YOLOXDataset, build_dataloader
from data_transforms.transforms import build_transforms

# Build dataset (no augmentation for preview)
preview_transforms = build_transforms(data_config, augment=False)
train_dataset = YOLOXDataset(
    config=data_config,
    split="train",
    transforms=preview_transforms,
)

print(f"Training samples: {len(train_dataset)}")
print(f"Classes: {train_dataset.class_names}")
print(f"Number of classes: {train_dataset.num_classes}")

In [ ]:
# Class distribution
class_counts = train_dataset.get_class_distribution()
print("Class distribution:")
for cls_name, count in class_counts.items():
    print(f"  {cls_name}: {count:,} instances")

# Plot distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(class_counts.keys(), class_counts.values(), color=["#e74c3c", "#95a5a6"])
ax.set_title("Fire Dataset — Class Distribution")
ax.set_ylabel("Instance Count")
for i, (cls, cnt) in enumerate(class_counts.items()):
    ax.text(i, cnt + 500, f"{cnt:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Display sample images with bounding boxes
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
indices = np.random.choice(len(train_dataset), size=8, replace=False)

for ax, idx in zip(axes.flat, indices):
    img, targets, _ = train_dataset[idx]
    # Convert from tensor to numpy for display
    if hasattr(img, "numpy"):
        img = img.numpy().transpose(1, 2, 0)
    img = np.clip(img, 0, 1) if img.max() <= 1.0 else np.clip(img / 255.0, 0, 1)
    ax.imshow(img)
    # Draw bounding boxes
    for target in targets:
        cls_id, cx, cy, w, h = target[:5]
        color = "red" if int(cls_id) == 0 else "gray"  # fire=red, smoke=gray
        x1, y1 = cx - w / 2, cy - h / 2
        rect = plt.Rectangle((x1, y1), w, h, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
    ax.set_title(f"Sample #{idx}")
    ax.axis("off")

plt.suptitle("Fire Dataset — Sample Images", fontsize=14)
plt.tight_layout()
plt.show()

## Training Configuration Review

Verify all hyperparameters before launching training.

In [ ]:
# Print training config summary
print("=" * 60)
print("TRAINING CONFIGURATION — Fire Detection")
print("=" * 60)
print(f"Model architecture : {train_config.get('arch', 'yolox-m')}")
print(f"Input size         : {train_config.get('input_size', 640)}")
print(f"Batch size         : {train_config.get('batch_size', 64)}")
print(f"Epochs             : {train_config.get('epochs', 300)}")
print(f"Learning rate      : {train_config.get('lr', 0.01)}")
print(f"LR scheduler       : {train_config.get('scheduler', 'cosine')}")
print(f"Warmup epochs      : {train_config.get('warmup_epochs', 5)}")
print(f"Weight decay       : {train_config.get('weight_decay', 5e-4)}")
print(f"FP16               : {train_config.get('fp16', True)}")
print(f"EMA                : {train_config.get('ema', True)}")
print("---")
print(f"Mosaic             : {train_config.get('mosaic', True)}")
print(f"Mixup              : {train_config.get('mixup', 0.5)}")
print(f"HSV augmentation   : {train_config.get('hsv_aug', True)}")
print(f"Flip               : {train_config.get('flip', 0.5)}")
print("---")
print(f"Pretrained weights : {train_config.get('pretrained', 'yolox_m.pth')}")
print(f"Num classes        : {data_config.get('num_classes', 2)}")
print(f"Output dir         : {train_config.get('output_dir', 'runs/fire')}")
print("=" * 60)

## Training

Launch YOLOX-M training on the fire detection dataset.  
Training progress is logged to W&B (if configured) and saved locally.

In [ ]:
from training.trainer import DetectionTrainer

# Create trainer with config
trainer = DetectionTrainer(
    config_path=PIPELINE_ROOT / "configs" / "training" / "fire.yaml",
    overrides={
        "device": str(device),
    },
)

print(f"Trainer initialized: {trainer}")
print(f"Output directory: {trainer.output_dir}")
print(f"\nStarting training...")

# Train — this will take several hours on a single GPU
history = trainer.train()

## Training Curves

Visualize loss and mAP progression across epochs.

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(history["epoch"], history["train_loss"], label="Train Loss", color="#e74c3c")
if "val_loss" in history:
    axes[0].plot(history["epoch"], history["val_loss"], label="Val Loss", color="#3498db")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# mAP curve
if "val_map50" in history:
    axes[1].plot(history["epoch"], history["val_map50"], label="mAP@0.5", color="#2ecc71")
    axes[1].axhline(y=0.85, color="red", linestyle="--", alpha=0.5, label="Target (0.85)")
    axes[1].axhline(y=0.75, color="orange", linestyle="--", alpha=0.5, label="Min (0.75)")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("mAP@0.5")
    axes[1].set_title("Validation mAP@0.5")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# Learning rate schedule
if "lr" in history:
    axes[2].plot(history["epoch"], history["lr"], color="#9b59b6")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("LR Schedule")
    axes[2].grid(True, alpha=0.3)

plt.suptitle("Fire Detection — Training Curves", fontsize=14)
plt.tight_layout()
plt.show()

# Print best metrics
if "val_map50" in history:
    best_epoch = np.argmax(history["val_map50"])
    print(f"\nBest epoch: {history['epoch'][best_epoch]}")
    print(f"Best mAP@0.5: {history['val_map50'][best_epoch]:.4f}")

## Quick Evaluation

Load the best checkpoint and evaluate on the validation set.

In [ ]:
# Evaluate best model on validation set
metrics = trainer.evaluate(split="val")

print("=" * 60)
print("VALIDATION RESULTS — Fire Detection")
print("=" * 60)
print(f"mAP@0.5       : {metrics.get('map50', 0):.4f}  (target: >= 0.85)")
print(f"mAP@0.5:0.95  : {metrics.get('map50_95', 0):.4f}")
print(f"Precision      : {metrics.get('precision', 0):.4f}  (target: >= 0.90)")
print(f"Recall         : {metrics.get('recall', 0):.4f}  (target: >= 0.88)")
print("---")

# Per-class breakdown
if "per_class" in metrics:
    print("\nPer-class results:")
    for cls_name, cls_metrics in metrics["per_class"].items():
        print(f"  {cls_name}:")
        print(f"    AP@0.5     : {cls_metrics.get('ap50', 0):.4f}")
        print(f"    Precision  : {cls_metrics.get('precision', 0):.4f}")
        print(f"    Recall     : {cls_metrics.get('recall', 0):.4f}")

# Status check
map50 = metrics.get("map50", 0)
if map50 >= 0.85:
    print("\n[PASS] mAP@0.5 meets target. Proceed to export.")
elif map50 >= 0.75:
    print("\n[WARN] mAP@0.5 above minimum but below target. Consider more training or data.")
else:
    print("\n[FAIL] mAP@0.5 below minimum. Review data quality, augmentation, or escalate model.")

## Save & Next Steps

### Saved Artifacts

- **Best weights**: `runs/fire/best.pth`
- **Last weights**: `runs/fire/last.pth`
- **Training log**: `runs/fire/train_log.json`
- **W&B run**: Check your W&B dashboard for full metrics and artifacts

### Next Steps

1. **If mAP >= 0.85**: Export to ONNX via `pipeline/04_evaluation/` and proceed to edge testing
2. **If 0.75 <= mAP < 0.85**: Run error analysis with FiftyOne, check for label noise, add factory data
3. **If mAP < 0.75**: Escalate to YOLOX-L (larger backbone) per escalation path

### Export Command

```bash
python core/p09_export/export.py --model runs/fire/best.pt --training-config features/safety-fire_detection/configs/06_training.yaml
```

### Model Naming Convention

- `.pth`: `fire_yoloxm_640_v{N}.pth` (original, for retraining)
- `.onnx`: `fire_yoloxm_640_v{N}.onnx` (edge deployment)